Author: Mike Bollinger
    Based on a framework by Logan Kline
    With editing and formatting support from ChatGPT v4.0
Created: 12/2/2024
Description: Integrates Survey123 with Smartsheet to update Smartsheet rows with S123 input

In [8]:
import smartsheet
import pandas as pd
from arcgis.gis import GIS
from datetime import datetime

Start by pulling down the Survey123 data and formatting it as it'll need to be

In [9]:
print("ArcGIS Online Org account")
gis = GIS('home')
print("Logged in to " + str(gis.properties.name) + " as " + str(gis.properties.user.username))

ArcGIS Online Org account
Logged in to NOAA GeoPlatform as mike.bollinger_noaa


Survey 123 configuration and formatting to fit Smartsheet format

In [10]:
#access and save layers from AGOL
s123_divedata = gis.content.get('ca90818eee7343f88e9ff02d0f0a9dd3').layers[1] #NCCOS UMS AUV database point layer

#convert to databases
s123_divedata_dbf = pd.DataFrame(s123_divedata.query().sdf)

#s123_divedata_dbf.head()

In [11]:
# Extract 'x' and 'y' values from the SHAPE column and add them to the existing DataFrame
s123_divedata_dbf[['x', 'y']] = s123_divedata_dbf['SHAPE'].apply(lambda shape: pd.Series([shape['x'], shape['y']]) if isinstance(shape, dict) else pd.Series([None, None]))

#s123_divedata_dbf.head()
print(s123_divedata_dbf)

    objectid                              globalid note0  \
0          1  51d7edd0-3c4b-48bd-8a35-333c739e8864  <NA>   
1          2  58e16d56-498c-4931-a3a5-e01a5d28a794  <NA>   
2          3  b9690502-061b-4d45-afdc-8e794a976663  <NA>   
3          4  7d895e3f-a4f6-4a8d-8f48-1451859c9a7f  <NA>   
4          5  f64dda7f-ba84-4d88-a3bc-2f706a9fbeae  <NA>   
5          6  ed8f5536-a117-47a0-acfa-dfbf5ba71caa  <NA>   
6          7  cd09c3c9-b5cf-4086-8773-acab6802c922  <NA>   
7          8  e824f4a2-b979-49a1-a32f-92945e9339cc  <NA>   
8          9  b593d86f-d5af-46cf-aee5-74e12a466055  <NA>   
9         10  ae4a40f1-c747-472a-8dae-755acd9e65da  <NA>   
10        11  c7394339-2258-455d-9fbb-1bdcc661be8a  <NA>   
11        12  0b3c4700-288a-48d3-8187-78e0e1ec6e9b  <NA>   
12        13  ba54ba08-904b-411e-88bd-86ae5125524a  <NA>   
13        14  c0521c60-e2d6-4566-aa88-1dd88f04297f  <NA>   
14        15  c0148d14-61fd-4353-9df7-7cd02ec1016a  <NA>   
15        16  86edf679-0592-48e2-9d58-20

In [12]:
#Subset survey123 database into fields that smartsheet accepts
s123_divedata_dbf_sub = s123_divedata_dbf.loc[:,("globalid","Project","CruiseID","StudyRegion","VesselPlatform","Vehicle","Pilot","Crew","DeployID","SiteID","DeployDate","DeployTime","y","x","SurfaceTime","DeploymentNotes","DataVolume")]

s123_divedata_dbf_sub = s123_divedata_dbf_sub.reset_index(drop=True)

#s123_divedata_dbf_sub.head()

In [13]:
#Rename columns to match Smartsheet
s123_divedata_dbf_reset = s123_divedata_dbf_sub.rename(columns={"globalid":"globalid","Project": "Project", "CruiseID":"Mission","StudyRegion":"Location","VesselPlatform":"Platform","Vehicle": "Vehicle","Pilot":"Pilot","Crew":"Team","DeployDate":"Dive Date","DeployID":"Dive Number","SiteID":"Site","DeployTime":"Mission Start Time","y":"Start Latitude","x":"Start Longitude","SurfaceTime":"End Time","DeploymentNotes":"Dive Description", "DataVolume":"Data Volume"})

s123_divedata_dbf_reset.reset_index(drop=True, inplace=True)

s123_divedata_dbf_reset.head()

,globalid,Project,Mission,Location,Platform,Vehicle,Pilot,Team,Dive Number,Site,Dive Date,Mission Start Time,Start Latitude,Start Longitude,End Time,Dive Description,Data Volume
0,51d7edd0-3c4b-48bd-8a35-333c739e8864,Engineering,ENG,NOAA Beaufort Lab,NCCOS R/V Tide,"NemoSens - ""Jacques""",Mike Bollinger,BrianDegan,ENG_uUUV_DIVE001,Shack,08/15/2024,18:37:29:126,34.657059,-76.640658,18:37:29:173,None,<NA>
1,58e16d56-498c-4931-a3a5-e01a5d28a794,Engineering,ENG,NOAA Beaufort Lab,NCCOS R/V Tide,"NemoSens - ""P.Sherman""",Mike Bollinger,BrianDegan,ENG_uUUV_DIVE002,Shack,08/15/2024,18:38:51:653,34.657342,-76.640585,18:38:51:692,<NA>,<NA>
2,b9690502-061b-4d45-afdc-8e794a976663,Engineering,ENG,NOAA Beaufort Lab,NCCOS R/V Tide,"NemoSens - ""P.Sherman""",Mike Bollinger,"BrianDegan,QuinWalker",ENG_uUUV_DIVE003,Shack,08/15/2024,18:38:54:284,34.657433,-76.640902,18:38:57:233,<NA>,<NA>
3,7d895e3f-a4f6-4a8d-8f48-1451859c9a7f,FL Outer Reef Bar GV,GM2401,FloridaAtlantic,FWC R/V Goldline,"NemoSens - ""P.Sherman""",Mike Bollinger,ChrisTaylor,GM2401_uUUV_DIVE001,Eng,08/22/2024,10:09:39:896,26.936908,-80.003302,10:21:43:999,<NA>,<NA>
4,f64dda7f-ba84-4d88-a3bc-2f706a9fbeae,FL Outer Reef Bar GV,GM2401,FloridaAtlantic,FWC R/V Goldline,"NemoSens - ""P.Sherman""",Mike Bollinger,ChrisTaylor,GM2401_uUUV_DIVE002,InnerGV,08/22/2024,10:40:15:617,26.937254,-80.015280,11:15:03:192,<NA>,0


In [14]:
##Cleaning up some smaller formatting issues to fine tune the export to smartsheet
# Convert "Mission Start Time" and "End Time" to HH:MM format
s123_divedata_dbf_reset['Mission Start Time'] = pd.to_datetime(s123_divedata_dbf_reset['Mission Start Time'], format='%H:%M:%S:%f').dt.strftime('%H:%M')
s123_divedata_dbf_reset['End Time'] = pd.to_datetime(s123_divedata_dbf_reset['End Time'], format='%H:%M:%S:%f').dt.strftime('%H:%M')
#Date formatting error
s123_divedata_dbf_reset['Dive Date'] = s123_divedata_dbf_reset['Dive Date'].apply(lambda x: datetime.strptime(x, "%m/%d/%Y").strftime("%Y-%m-%d"))
#FloridaAtlantic sync error
s123_divedata_dbf_reset['Location'] = s123_divedata_dbf_reset['Location'].replace("FloridaAtlantic", "Florida Atlantic")
#Mission Iconic Reef sync error
s123_divedata_dbf_reset['Project'] = s123_divedata_dbf_reset['Project'].replace("M:IR", "Mission: Iconic Reefs")
#BOEM - Carolina Long Bay sync error
s123_divedata_dbf_reset['Project'] = s123_divedata_dbf_reset['Project'].replace("BOEM-CLB", "BOEM: Carolina Long Bay")

#s123_divedata_dbf_reset.head()

In [15]:
#Isolate out the global IDs from s123 so that it can be compared with SMartsheet to avoid duplicates
s123_divedata_globalid = s123_divedata_dbf_reset['globalid'].tolist()
print(s123_divedata_globalid)

['51d7edd0-3c4b-48bd-8a35-333c739e8864', '58e16d56-498c-4931-a3a5-e01a5d28a794', 'b9690502-061b-4d45-afdc-8e794a976663', '7d895e3f-a4f6-4a8d-8f48-1451859c9a7f', 'f64dda7f-ba84-4d88-a3bc-2f706a9fbeae', 'ed8f5536-a117-47a0-acfa-dfbf5ba71caa', 'cd09c3c9-b5cf-4086-8773-acab6802c922', 'e824f4a2-b979-49a1-a32f-92945e9339cc', 'b593d86f-d5af-46cf-aee5-74e12a466055', 'ae4a40f1-c747-472a-8dae-755acd9e65da', 'c7394339-2258-455d-9fbb-1bdcc661be8a', '0b3c4700-288a-48d3-8187-78e0e1ec6e9b', 'ba54ba08-904b-411e-88bd-86ae5125524a', 'c0521c60-e2d6-4566-aa88-1dd88f04297f', 'c0148d14-61fd-4353-9df7-7cd02ec1016a', '86edf679-0592-48e2-9d58-20622b8468ec', '82998fb0-92c8-4128-9bed-a15d84fc3ad2', '46feb4f5-5517-44ed-a84e-e8eb27d3e438', 'cd50cf7d-e73f-47db-b0f2-85fd572204d4', '904e93b1-646a-4b9e-bc77-92e15cb518e4', 'c0837e67-ba6e-4bd1-ab00-c374541bfac7', '26a4b8d5-f5fa-435c-884e-a12d593c012f', '57cd0bc6-e90f-460a-bc57-1fcc64f16428', 'ec1b83b9-f1ab-44b8-bb52-f36dc5158169', '97c598b4-bdc8-48fe-9199-7421afcdf983',

Start working in Smartsheet

In [18]:
# Set up Smartsheet access
access_token = '6H72JtOX0pVZaTumEofA0zla9X1stAruyhHBz' #'--PUT ACCESS TOKEN HERE--'
smartSheet_Client = smartsheet.Smartsheet(access_token, api_base=smartsheet.__gov_base__)
smartSheet_Client.errors_as_exceptions(True)
divelog_sheetid = '3136216628520844'



# Function to get the maximum Edit Date from Smartsheet
def get_max_edit_date(sheet, edit_date_column_id):
    max_edit_date = None
    for row in sheet.rows:
        for cell in row.cells:
            if cell.column_id == edit_date_column_id and cell.value:
                edit_date = datetime.strptime(cell.value, '%Y-%m-%dT%H:%M:%SZ').date()
                if not max_edit_date or edit_date > max_edit_date:
                    max_edit_date = edit_date
    return max_edit_date

In [19]:
#find the most recent smartsheet edit
client = smartSheet_Client
sheet = client.Sheets.get_sheet(divelog_sheetid)

# Function to get the column ID map
def get_column_id_map(sheet):
    return {col.title: col.id for col in sheet.columns}

column_id_map = get_column_id_map(sheet)
edit_date_column_id = column_id_map.get('Edit Date')

if not edit_date_column_id:
    print("Edit Date column not found.")

max_edit_date = get_max_edit_date(sheet, edit_date_column_id)
if not max_edit_date:
    print("No edit date found in the Smartsheet.")

print(f"Max Edit Date: {max_edit_date}")

Max Edit Date: 2024-12-02


In [20]:
#make a list of all smartsheet global ids to compare with survey123 to avoid duplicates
COLUMN_NAME = "globalid"  # Replace with the desired column name

# Find the column ID based on the column name
column_id = None
for column in sheet.columns:
    if column.title == COLUMN_NAME:
        column_id = column.id
        break

if column_id is None:
    print(f"Column '{COLUMN_NAME}' not found in the sheet.")
else:
    # Extract populated values from the specified column
    smartsheet_divedata_globalids = []
    for row in sheet.rows:
        for cell in row.cells:
            if cell.column_id == column_id and cell.value is not None:
                smartsheet_divedata_globalids.append(cell.value)
                break  # Move to the next row after finding a populated value

    # Display the populated values list (you can now compare this list with another)
    print(f"Populated values in column '{COLUMN_NAME}':")
    print(smartsheet_divedata_globalids)

Populated values in column 'globalid':
['51d7edd0-3c4b-48bd-8a35-333c739e8864', '58e16d56-498c-4931-a3a5-e01a5d28a794', 'b9690502-061b-4d45-afdc-8e794a976663', '7d895e3f-a4f6-4a8d-8f48-1451859c9a7f', 'f64dda7f-ba84-4d88-a3bc-2f706a9fbeae', 'ed8f5536-a117-47a0-acfa-dfbf5ba71caa', 'cd09c3c9-b5cf-4086-8773-acab6802c922', 'e824f4a2-b979-49a1-a32f-92945e9339cc', 'b593d86f-d5af-46cf-aee5-74e12a466055', 'ae4a40f1-c747-472a-8dae-755acd9e65da', 'c7394339-2258-455d-9fbb-1bdcc661be8a', '0b3c4700-288a-48d3-8187-78e0e1ec6e9b', 'ba54ba08-904b-411e-88bd-86ae5125524a', 'c0521c60-e2d6-4566-aa88-1dd88f04297f', 'c0148d14-61fd-4353-9df7-7cd02ec1016a', '86edf679-0592-48e2-9d58-20622b8468ec', '46feb4f5-5517-44ed-a84e-e8eb27d3e438', '82998fb0-92c8-4128-9bed-a15d84fc3ad2', 'cd50cf7d-e73f-47db-b0f2-85fd572204d4', '904e93b1-646a-4b9e-bc77-92e15cb518e4', 'c0837e67-ba6e-4bd1-ab00-c374541bfac7', '26a4b8d5-f5fa-435c-884e-a12d593c012f', '57cd0bc6-e90f-460a-bc57-1fcc64f16428', 'ec1b83b9-f1ab-44b8-bb52-f36dc5158169', 

In [22]:
#compare lists and cut duplicates
newdivedata = s123_divedata_dbf_reset[~s123_divedata_dbf_reset['globalid'].isin(smartsheet_divedata_globalids)]

print(newdivedata)

                                globalid                  Project Mission  \
0   51d7edd0-3c4b-48bd-8a35-333c739e8864              Engineering     ENG   
1   58e16d56-498c-4931-a3a5-e01a5d28a794              Engineering     ENG   
2   b9690502-061b-4d45-afdc-8e794a976663              Engineering     ENG   
3   7d895e3f-a4f6-4a8d-8f48-1451859c9a7f     FL Outer Reef Bar GV  GM2401   
4   f64dda7f-ba84-4d88-a3bc-2f706a9fbeae     FL Outer Reef Bar GV  GM2401   
5   ed8f5536-a117-47a0-acfa-dfbf5ba71caa     FL Outer Reef Bar GV  GM2401   
6   cd09c3c9-b5cf-4086-8773-acab6802c922     FL Outer Reef Bar GV  GM2401   
7   e824f4a2-b979-49a1-a32f-92945e9339cc     FL Outer Reef Bar GV  GM2401   
8   b593d86f-d5af-46cf-aee5-74e12a466055     FL Outer Reef Bar GV  GM2401   
9   ae4a40f1-c747-472a-8dae-755acd9e65da    Mission: Iconic Reefs  MIR241   
10  c7394339-2258-455d-9fbb-1bdcc661be8a    Mission: Iconic Reefs  MIR241   
11  0b3c4700-288a-48d3-8187-78e0e1ec6e9b    Mission: Iconic Reefs  MIR241   

In [23]:
## Functions to complete the processing of the dataframe and upload to Smartsheet

# Function to prepare Smartsheet rows
def prepare_smartsheet_rows(data, column_id_map, extra_columns):
    new_rows = []
    for _, item in data.iterrows():  # Use iterrows to loop through DataFrame rows
        row = smartsheet.models.Row()
        row.to_bottom = True
        for column, value in item.items():
            if column in column_id_map and pd.notna(value):  # Check for missing values
                row.cells.append({'column_id': column_id_map[column], 'value': value})
            elif column in column_id_map:
                row.cells.append({'column_id': column_id_map[column], 'value': ''})  # Use an empty string if NA
        if extra_columns:
            for column, value in extra_columns.items():
                row.cells.append({'column_id': column_id_map[column], 'value': value})
        new_rows.append(row)
    return new_rows

# Function to add rows to Smartsheet
def add_rows_to_smartsheet(client, sheet_id, rows):
    try:
        response = client.Sheets.add_rows(sheet_id, rows)
        print(f"Uploaded {len(rows)} rows successfully!")
    except smartsheet.exceptions.ApiError as e:
        print(f"Error adding rows to Smartsheet: {e}")

# Main function to process each sheet
def process_sheet(client, sheet_info, data):
    # Retrieve the sheet
    sheet = client.Sheets.get_sheet(sheet_info['sheet_id'])
    # Get the column ID map
    column_id_map = get_column_id_map(sheet)

    # Prepare rows for Smartsheet
    rows = prepare_smartsheet_rows(data, column_id_map, sheet_info.get('extra_columns'))
    # Add rows to Smartsheet
    add_rows_to_smartsheet(client, sheet_info['sheet_id'], rows)

# Define the information for each sheet
sheets_info = [
    {
        'sheet_id': '3136216628520844',
        'fields_map': {
            'Project': 'Project',
            'Mission': 'Mission',
            'Location': 'Location',
            'Platform': 'Platform',
            'Vehicle': 'Vehicle',
            'Dive Date': 'Dive Date',
            'Dive Number': 'Dive Number',
            'Site': 'Site',
            'Mission Start Time': 'Mission Start Time',
            'Start Latitude': 'Start Latitude',
            'Start Longitude': 'Start Longitude',
            'End Time': 'End Time',
            'Dive Description': 'Dive Description',
            'Data Volume':'Data Volume',
            'globalid': 'globalid'
        }
    }
]

# Run the processing for each sheet
for sheet_info in sheets_info:
    # Map DataFrame columns based on 'fields_map'
    data_to_upload = newdivedata[list(sheet_info['fields_map'].keys())]
    data_to_upload.rename(columns=sheet_info['fields_map'], inplace=True)
    process_sheet(smartSheet_Client, sheet_info, data_to_upload)


C:\Users\Mike.Bollinger\AppData\Local\Temp\1\ipykernel_4656\460037343.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_to_upload.rename(columns=sheet_info['fields_map'], inplace=True)


Uploaded 43 rows successfully!
